In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import RobustScaler, OrdinalEncoder, OneHotEncoder
from sklearn.metrics import classification_report

c:\Users\User\.conda\envs\default_env\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [1]:
import pickle
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

class LoanModel:
    def __init__(self, df):
        self.df = df.copy()
        self.model = None
        self.ord_encoder = None
        self.onehot_encoder = None
        self.scaler = None
        self.feature_list = None

    def preprocess(self):
        # Fix gender
        self.df['person_gender'] = self.df['person_gender'].str.lower().str.replace(' ', '')
        self.df['person_gender'] = self.df['person_gender'].replace({'male': 'Male', 'female': 'Female'})

        # Encode previous defaults
        self.df['previous_loan_defaults_on_file'] = self.df['previous_loan_defaults_on_file'].map({'Yes': 1, 'No': 0}).astype(int)

        # Fill missing income with median by class
        self.df['person_income'] = self.df.groupby('loan_status')['person_income'].transform(lambda x: x.fillna(x.median()))

        # Encode education (ordinal)
        education_order = [['High School', 'Associate', 'Bachelor', 'Master', 'Doctorate']]
        self.ord_encoder = OrdinalEncoder(categories=education_order)
        self.df['person_education'] = self.ord_encoder.fit_transform(self.df[['person_education']]).astype(int)

        # One-hot encoding
        onehot_cols = ['person_gender', 'person_home_ownership', 'loan_intent']
        self.onehot_encoder = OneHotEncoder(sparse=False, drop='first')
        onehot_array = self.onehot_encoder.fit_transform(self.df[onehot_cols])
        onehot_df = pd.DataFrame(onehot_array, columns=self.onehot_encoder.get_feature_names_out(onehot_cols), index=self.df.index)

        # Drop and join
        self.df.drop(columns=onehot_cols, inplace=True)
        self.df = pd.concat([self.df, onehot_df], axis=1)

    def split_and_scale(self):
        X = self.df.drop('loan_status', axis=1)
        y = self.df['loan_status']
        self.numerical_cols = ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt',
                               'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

        # Scaling
        self.scaler = RobustScaler()
        X_train[self.numerical_cols] = self.scaler.fit_transform(X_train[self.numerical_cols])
        X_test[self.numerical_cols] = self.scaler.transform(X_test[self.numerical_cols])

        self.feature_list = X_train.columns.tolist()
        self.X_train, self.X_test = X_train, X_test
        self.y_train, self.y_test = y_train, y_test

    def train_model(self):
        model = XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            subsample=1,
            colsample_bytree=1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss',
            scale_pos_weight=self.y_train.value_counts()[0] / self.y_train.value_counts()[1]
        )
        model.fit(self.X_train, self.y_train)
        self.model = model

    def evaluate_model(self):
        y_pred = self.model.predict(self.X_test)
        report = classification_report(self.y_test, y_pred, target_names=['0', '1'])
        print("Classification Report:\n", report)
        return report

    def save_all(self, model_path='best_model.pkl', scaler_path='scaler.pkl',
                 ordinal_path='ordinal_encoder.pkl', onehot_path='onehot_encoder.pkl', feature_path='feature_list.pkl'):
        with open(model_path, 'wb') as f:
            pickle.dump(self.model, f)
        with open(scaler_path, 'wb') as f:
            pickle.dump(self.scaler, f)
        with open(ordinal_path, 'wb') as f:
            pickle.dump(self.ord_encoder, f)
        with open(onehot_path, 'wb') as f:
            pickle.dump(self.onehot_encoder, f)
        with open(feature_path, 'wb') as f:
            pickle.dump(self.feature_list, f)


c:\Users\User\.conda\envs\default_env\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
df_raw = pd.read_csv('Dataset_A_loan.csv')
loan_pipeline = LoanModel(df_raw)
loan_pipeline.preprocess()

c:\Users\User\.conda\envs\default_env\lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [3]:
loan_pipeline.split_and_scale()

In [4]:
loan_pipeline.train_model()

In [5]:
loan_pipeline.evaluate_model()

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.92      0.95      7000
           1       0.76      0.91      0.83      2000

    accuracy                           0.92      9000
   macro avg       0.87      0.92      0.89      9000
weighted avg       0.93      0.92      0.92      9000



'              precision    recall  f1-score   support\n\n           0       0.97      0.92      0.95      7000\n           1       0.76      0.91      0.83      2000\n\n    accuracy                           0.92      9000\n   macro avg       0.87      0.92      0.89      9000\nweighted avg       0.93      0.92      0.92      9000\n'

In [7]:
loan_pipeline.save_all()